# Stage 5: Machine Learning Model Training and Comparison
## Project: AI-Based Student Dropout Prediction and Early Warning System

**Objective:** Train, evaluate, and compare 3 machine learning algorithms for Early Warning dropout prediction:
1. **Logistic Regression** (Interpretable linear model)
2. **Decision Tree Classifier** (Rule-based tree)
3. **Random Forest Classifier** (Ensemble of bagged trees)

> **Early Warning Constraint:**
> - We predict dropout risk using information available **by the end of the first semester**.
> - All 6 second-semester features are excluded to ensure zero future/temporal data leakage.
> - `class_weight='balanced'` is used across all models to handle the 2.1:1 class distribution.

In [ ]:
import os
import json
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve
)

sns.set_theme(style="whitegrid", palette="muted")

# 1. Load preprocessed train and test data
train_path = os.path.join("..", "dataset", "train_data.csv")
test_path = os.path.join("..", "dataset", "test_data.csv")
meta_path = os.path.join("..", "models", "feature_metadata.json")

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

with open(meta_path, "r", encoding="utf-8") as f:
    meta = json.load(f)

# Exclude 2nd-semester features, Target, and Dropout_Risk from X
second_sem_cols = meta.get("second_sem_features", [])
drop_cols = list(second_sem_cols) + ["Dropout_Risk"]
if "Target" in train_df.columns:
    drop_cols.append("Target")

X_train = train_df.drop(columns=[c for c in drop_cols if c in train_df.columns])
y_train = train_df["Dropout_Risk"]
X_test = test_df.drop(columns=[c for c in drop_cols if c in test_df.columns])
y_test = test_df["Dropout_Risk"]

print(f"Early warning features count: {X_train.shape[1]}")
print(f"X_train shape: {X_train.shape} | X_test shape: {X_test.shape}")

### 2. Define and Train the 3 Models in Scikit-Learn Pipelines
Pipelines prevent data leakage by ensuring `StandardScaler` is fitted strictly on `X_train`.

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
    ]),
    "Decision Tree": Pipeline([
        ("scaler", StandardScaler()),
        ("model", DecisionTreeClassifier(max_depth=6, min_samples_split=10, class_weight="balanced", random_state=42))
    ]),
    "Random Forest": Pipeline([
        ("scaler", StandardScaler()),
        ("model", RandomForestClassifier(n_estimators=150, max_depth=12, min_samples_split=5, class_weight="balanced", random_state=42))
    ])
}

results = {}
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]
    
    results[name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Dropout Recall": recall_score(y_test, y_pred),
        "Dropout Precision": precision_score(y_test, y_pred),
        "Dropout F1-Score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob),
        "Confusion Matrix": confusion_matrix(y_test, y_pred)
    }

summary_df = pd.DataFrame(results).T
display(summary_df.round(4))

### 3. Confusion Matrix Visualizations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for idx, (name, res) in enumerate(results.items()):
    sns.heatmap(res["Confusion Matrix"], annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[idx])
    axes[idx].set_title(f"{name}\nRecall: {res['Dropout Recall']*100:.1f}%")
    axes[idx].set_xlabel("Predicted Class")
    axes[idx].set_ylabel("Actual Class")
plt.show()

### 4. Random Forest Feature Importance

In [ ]:
rf_model = models["Random Forest"].named_steps["model"]
feat_imp = pd.Series(rf_model.feature_importances_, index=X_train.columns).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=feat_imp.head(15), y=feat_imp.head(15).index, palette="viridis")
plt.title("Top 15 Early-Warning Features (Random Forest)")
plt.xlabel("Importance")
plt.show()